**EEG–Eye-Tracking Multimodal Research | Presented by: Reza Saadatyar (2026)**

**04 | Unimodal Baselines and Classical Fusion**

![Unimodal Baselines and Classical Fusion](../../images/04_unimodal_baselines_and_classical_fusion.svg)

**Objective:** compare task prior, gaze-only, EEG-only and simple
       EEG+gaze fusion methods on the identical paired windows. **Input:**
       Stage-03 feature table with frozen person roles. **No Test scoring:**
       estimators are fitted on Train and compared on Validation only.
       Features are derived before fitting; imputation and scaling are
       trained exclusively inside Train-fitted estimator pipelines.

       A model may classify reading conditions partly through stimulus or
       run differences; high accuracy cannot be interpreted as comprehension.

**00. Colab Repository Setup**

In [ ]:
# ==========================00. Colab Repository Setup=========================

from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
PROJECT = "eeg_eye_tracking_fusion"
repo_root = Path('/content/Eye-Tracking-Neurocognitive-Intelligence')
if not Path('/content').is_dir():
    raise RuntimeError('These notebooks target Google Colab.')
if not (repo_root / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL,
                    str(repo_root)], check=True)
work = repo_root / PROJECT
output_dir = work / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
raw_dir = Path('/content/zuco2_source_cache')
raw_dir.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(work / 'code' / 'src'))
print('Repository:', repo_root)
print('Project:', work)
print('Outputs:', output_dir)
print('Source cache (temporary):', raw_dir)


**01. Imports and Research Environment**

In [ ]:
# =====================01. Imports and Research Environment====================

import importlib.util
import json
import random
import subprocess
import sys
import time
from pathlib import Path

if importlib.util.find_spec('mne') is None:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                    'mne>=1.6,<2'], check=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

SEED = 17
random.seed(SEED)
np.random.seed(SEED)
print('Python:', sys.version.split()[0])
print('Free disk (GiB):', round(
    __import__('shutil').disk_usage('/content').free / 2**30, 2))
print('NumPy:', np.__version__, 'pandas:', pd.__version__)


**02. Load Paired Feature Table and Fixed Labels**

One row corresponds to one non-overlapping, paired 12-s window.
          Participant-disjoint roles rather than window randomisation
          determine the development and final evaluation partitions.

In [ ]:
# ================02. Load Paired Feature Table and Fixed Labels===============

from models import (CLASSICAL_ORDER, CLASS_TO_INDEX, estimator,
                    matrix, metrics, probabilities,
                    record_predictions)
import joblib
from sklearn.metrics import ConfusionMatrixDisplay

table = pd.read_csv(output_dir / '03_features.csv')
if table.window_id.duplicated().any():
    raise ValueError('Duplicated paired window IDs')
if not table.groupby('participant_id').split.nunique().eq(1).all():
    raise ValueError('Person appears in multiple splits')
train = table.loc[table.split.eq('train')].copy()
valid = table.loc[table.split.eq('validation')].copy()
test = table.loc[table.split.eq('test')].copy()
if min(len(train), len(valid), len(test)) < 1:
    raise ValueError('Source split has an empty role')
y_train = train.task.map(CLASS_TO_INDEX).to_numpy()
y_valid = valid.task.map(CLASS_TO_INDEX).to_numpy()
print('Train:', len(train), 'Validation:', len(valid),
      'Reserved Test:', len(test))


**03. Fit Task Prior and Separate Unimodal Estimators**

The task-prior model has no gaze or EEG predictor.
          Logistic and forest models use fixed settings, with Train-only
          median imputation. Original source identifiers are excluded.

In [ ]:
# =============03. Fit Task Prior and Separate Unimodal Estimators=============

fitted, predictions, records = {}, {}, []
feature_group = {
    'class_prior': 'gaze',
    'gaze_logistic': 'gaze', 'eeg_logistic': 'eeg',
    'early_fusion_logistic': 'fused',
    'gaze_forest': 'gaze', 'eeg_forest': 'eeg',
}
for name in CLASSICAL_ORDER:
    if name == 'late_equal_fusion':
        continue
    group = feature_group[name]
    model = estimator(name, seed=SEED)
    x_train = matrix(train, group)
    x_val = matrix(valid, group)
    model.fit(x_train, y_train)
    proba = probabilities(model, x_val)
    fitted[name] = model
    predictions[name] = proba
    row = {'model': name, 'partition': 'validation',
           'n_windows': len(valid),
           'n_people': valid.participant_id.nunique(),
           **metrics(y_valid, proba.argmax(axis=1))}
    records.append(row)
    joblib.dump(model, output_dir / f'04_{name}.joblib')
    print(name, 'Macro F1:', round(row['macro_f1'], 4))


**04. Fit Early Fusion and Evaluate Fixed Late Fusion**

Early Fusion is already fitted as a combined feature vector.
          Late Fusion averages the two independent logistic probability
          vectors equally; its weight is fixed rather than tuned on Test.

In [ ]:
# =============04. Fit Early Fusion and Evaluate Fixed Late Fusion=============

late = 0.5 * (predictions['gaze_logistic'] +
              predictions['eeg_logistic'])
predictions['late_equal_fusion'] = late
records.append({
    'model': 'late_equal_fusion', 'partition': 'validation',
    'n_windows': len(valid),
    'n_people': valid.participant_id.nunique(),
    **metrics(y_valid, late.argmax(axis=1))
})
(output_dir / '04_late_fusion.json').write_text(
    json.dumps({'weight_eeg': 0.5, 'weight_gaze': 0.5,
                'components': ['eeg_logistic', 'gaze_logistic']},
               indent=2), encoding='utf-8')
summary = pd.DataFrame(records)
assert set(summary.model) == set(CLASSICAL_ORDER)
display(summary.sort_values('macro_f1', ascending=False))


**05. Compare Validation Metrics and a Confusion Matrix**

Validation is development evidence, not a Test estimate.
          Candidate selection across classical and neural models is deferred
          to Notebook 06 under a fixed ranking rule.

In [ ]:
# ============05. Compare Validation Metrics and a Confusion Matrix============

fig, ax = plt.subplots(figsize=(9, 4.5))
summary.sort_values('macro_f1').plot.barh(
    x='model', y='macro_f1', ax=ax, legend=False)
ax.set(xlabel='Validation fixed-label Macro F1',
       ylabel='Model', xlim=(0, 1.03),
       title='Classical and simple-fusion comparison')
fig.tight_layout()
fig.savefig(output_dir / '04_validation_comparison.png', dpi=170)
plt.show()
best = summary.sort_values(['macro_f1', 'balanced_accuracy'],
                           ascending=False).iloc[0].model
fig, ax = plt.subplots(figsize=(4.3, 4))
ConfusionMatrixDisplay.from_predictions(
    y_valid, predictions[best].argmax(axis=1),
    labels=[0, 1], display_labels=['NR', 'TSR'], ax=ax,
    colorbar=False)
ax.set_title(f'Descriptive Validation: {best}')
fig.tight_layout()
fig.savefig(output_dir / '04_validation_confusion.png', dpi=170)
plt.show()


**06. Check Probabilities and Save Model Evidence**

Predictions for every candidate must refer to the exact
          Validation window IDs. The saved models and tables are stage-owned
          inputs to the final primary selection in Notebook 06.

In [ ]:
# ===============06. Check Probabilities and Save Model Evidence===============

saved = []
for name in CLASSICAL_ORDER:
    proba = predictions[name]
    assert proba.shape == (len(valid), 2)
    assert np.isfinite(proba).all()
    assert np.allclose(proba.sum(axis=1), 1., atol=1e-5)
    saved.append(record_predictions(valid, name, proba))
prediction_table = pd.concat(saved, ignore_index=True)
prediction_table.to_csv(
    output_dir / '04_validation_predictions.csv', index=False)
summary.to_csv(output_dir / '04_validation_metrics.csv', index=False)
print('Validation rows:', len(prediction_table))
assert prediction_table.groupby('model').window_id.nunique().eq(
    len(valid)).all()


**07. Interpretation and Next Dependency**

The simple fusion gain over single modalities is an empirical
          question. This notebook does not rank advanced networks or access
          Test labels for evaluation. The next notebook trains compact
          temporal models using the same Train/Validation person roles.

In [ ]:
# ====================07. Interpretation and Next Dependency===================

print('Saved fitted classical pipelines:', len(fitted))
print('Next: 05_temporal_and_advanced_fusion.ipynb')
